In [41]:
# Task 4.1

import sqlite3

conn = sqlite3.connect('BUSROUTES.db')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS Service (
    ServiceNo TEXT PRIMARY KEY,
    Operator TEXT NOT NULL,
    Category TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS Stop (
    BusStopCode TEXT PRIMARY KEY,
    RoadName TEXT NOT NULL,
    Description TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS Route (
    ServiceNo TEXT,
    Operator TEXT,
    Direction INTEGER CHECK (Direction IN (1,2)),
    StopSequence INTEGER,
    BusStopCode TEXT,
    PRIMARY KEY (ServiceNo, Direction, StopSequence),
    FOREIGN KEY (ServiceNo) REFERENCES Service(ServiceNo),
    FOREIGN KEY (BusStopCode) REFERENCES Stop(BusStopCode)
)
''')

conn.commit()
conn.close()

In [42]:
# Task 4.2

import sqlite3
import csv

conn = sqlite3.connect('BUSROUTES.db')
cursor = conn.cursor()

with open('Service.csv', 'r') as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:
        cursor.execute('''
        INSERT OR REPLACE INTO Service (ServiceNo, Operator, Category)
        VALUES (?, ?, ?)
        ''', row)

with open('Stop.csv', 'r') as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:
        cursor.execute('''
        INSERT OR REPLACE INTO Stop (BusStopCode, RoadName, Description)
        VALUES (?, ?, ?)
        ''', row)

with open('Route.csv', 'r') as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:
        cursor.execute('''
        INSERT OR REPLACE INTO Route
        (ServiceNo, Operator, Direction, StopSequence, BusStopCode)
        VALUES (?, ?, ?, ?, ?)
        ''', row)

conn.commit()
conn.close()

In [43]:
# Task 4.3

import sqlite3

def query_service_routes(service_no):
    
    conn = sqlite3.connect('BUSROUTES.db')
    cursor = conn.cursor()

    cursor.execute('''
    SELECT Route.Direction, Route.StopSequence, Route.BusStopCode,
           Stop.RoadName, Stop.Description
    FROM Route
    JOIN Stop ON Route.BusStopCode = Stop.BusStopCode
    WHERE Route.ServiceNo = ?
    ORDER BY Route.Direction ASC, Route.StopSequence ASC
    ''', (service_no,))

    results = cursor.fetchall()
    
    conn.close()

    if not results:
        print(f"No routes found for service {service_no}")
        return

    current_direction = None
    for row in results:
        direction, seq, code, road, desc = row

        if direction != current_direction:
            print(f"\nDirection: {direction}")
            current_direction = direction

        print(f"\nStop Sequence: {seq}")
        print(f"Bus Stop Code: {code}")
        print(f"Road Name: {road}")
        print(f"Description: {desc}")

# Main
service_no = input("Enter bus service number: ")
query_service_routes(service_no)

Enter bus service number: 10

Direction: 1

Stop Sequence: 1
Bus Stop Code: 75009
Road Name: Tampines Ctrl 1
Description: Tampines Int

Stop Sequence: 2
Bus Stop Code: 76059
Road Name: Tampines Ave 5
Description: Opp Our Tampines Hub

Stop Sequence: 3
Bus Stop Code: 76069
Road Name: Tampines Ave 5
Description: Blk 147

Stop Sequence: 4
Bus Stop Code: 96289
Road Name: Simei Ave
Description: Changi General Hosp

Stop Sequence: 5
Bus Stop Code: 96109
Road Name: Simei Ave
Description: Opp Blk 3012

Stop Sequence: 6
Bus Stop Code: 85079
Road Name: Upp Changi Rd
Description: Flextronic

Stop Sequence: 7
Bus Stop Code: 85089
Road Name: Upp Changi Rd
Description: Aft Sungei Bedok

Stop Sequence: 8
Bus Stop Code: 85069
Road Name: Bedok Rd
Description: Bedok Mkt Pl

Stop Sequence: 9
Bus Stop Code: 85059
Road Name: Bedok Rd
Description: Bef Jln Chempaka Kuning

Stop Sequence: 10
Bus Stop Code: 85049
Road Name: Bedok Rd
Description: Opp Man Fatt Lam Tp

Stop Sequence: 11
Bus Stop Code: 85039
Road 